# 02 — Departure Feature Engineering (V9.0)

**Post-split: Lag → Aircraft Continuity → Target Encoding → Weather → V9.0新特征**

In [1]:
# 0. Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_curve
)
from sklearn.isotonic import IsotonicRegression
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import catboost as cb
import lightgbm as lgb
import shap
import joblib
import json
import os
import sys

# === Project paths (RCF: only change BASE_DIR) ===
BASE_DIR = Path('../../..')             # RCF: Path('/path/to/project')
if not (BASE_DIR / 'src').exists():
    BASE_DIR = Path('../../..')

DATA_RAW = BASE_DIR / 'data' / 'raw' / 'LGA_Dataset'
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'
MODEL_DIR = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs'
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Add src to path
sys.path.insert(0, str(BASE_DIR / 'src'))

from features.lag_features import add_lag_features, add_congestion_features, compute_v4_lag_features
from features.aircraft_features import compute_prev_aircraft_delay
from models.calibration import (
    fit_isotonic_calibration, apply_calibration, compute_ece
)
from models.temporal_weights import compute_temporal_weights, combine_temporal_and_class_weights

SEED = 42
np.random.seed(SEED)
print('Imports OK')

Imports OK


## 1. Load Data from NB01

In [2]:
# === Load data from 01 ===
import pickle
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'
DATA_RAW = BASE_DIR / 'data' / 'raw' / 'LGA_Dataset'
train = pd.read_parquet(DATA_PROCESSED / 'dep_train_raw.parquet')
test = pd.read_parquet(DATA_PROCESSED / 'dep_test_raw.parquet')
df_arr = pd.read_parquet(DATA_PROCESSED / 'dep_arrivals_for_continuity.parquet')
arr_train = pd.read_parquet(DATA_PROCESSED / 'dep_arr_train.parquet')
arr_test_ctx = pd.read_parquet(DATA_PROCESSED / 'dep_arr_test_ctx.parquet')

ctx = pickle.load(open(DATA_PROCESSED / 'dep_context.pkl', 'rb'))
cutoff = ctx['cutoff']
df_lga_wx = ctx.get('df_lga_wx')
df_dest_wx = ctx.get('df_dest_wx')
lga_wx_hourly = ctx.get('lga_wx_hourly')
dest_wx_hourly = ctx.get('dest_wx_hourly')
df_gdp = ctx.get('df_gdp')
df_gsp = ctx.get('df_gsp')
df_dd = ctx.get('df_dd')
df_ad = ctx.get('df_ad')
HAS_FAA = ctx.get('HAS_FAA', False)
HAS_LGA_WX = ctx.get('HAS_LGA_WX', False)
HAS_DEST_WX = ctx.get('HAS_DEST_WX', False)
HAS_ARRIVALS = df_arr is not None

# Fix Date column dtype
for d in [train, test]:
    if d['Date'].dtype == object:
        d['Date'] = pd.to_datetime(d['Date'])

print(f'Loaded: train={len(train)}, test={len(test)}')
print(f'Flags: FAA={HAS_FAA}, LGA_WX={HAS_LGA_WX}, DEST_WX={HAS_DEST_WX}')


Loaded: train=103366, test=44113
Flags: FAA=True, LGA_WX=True, DEST_WX=True


## 2. Departure Lag Features

In [3]:
# 2.1 Departure lag features (5 features)
# Reuse add_lag_features() with departure columns

print('--- Computing departure lag features ---')

# Train
train = add_lag_features(
    train,
    delay_col='Dep_Calculated_Delay',
    datetime_col='Scheduled_Departure_Datetime',
    date_col='Date',
    terminal_col='Terminal_Clean',
    verbose=True
)

# Test with context buffer (1 day from train tail)
context_date = test['Date'].min() - pd.Timedelta(days=1)
context = train[train['Date'] >= context_date].copy()
print(f'\nContext buffer: {len(context)} flights from {context_date}')

test_with_ctx = pd.concat([context, test], ignore_index=True)
test_with_ctx = add_lag_features(
    test_with_ctx,
    delay_col='Dep_Calculated_Delay',
    datetime_col='Scheduled_Departure_Datetime',
    date_col='Date',
    terminal_col='Terminal_Clean',
    verbose=False
)
# Filter back to test-only rows
test = test_with_ctx[test_with_ctx['Date'] > cutoff].copy()
print(f'Test after lag features: {len(test):,} rows')

--- Computing departure lag features ---
Lag Features Added:
  - delay_rolling_1h: mean=13.56, null_count=0
  - delay_rolling_3h: mean=11.85, null_count=0
  - delay_rate_1h: mean=0.20, null_count=0
  - severe_delay_count_prev: mean=6.74, null_count=0
  - terminal_delay_1h: mean=13.49, null_count=0

Context buffer: 541 flights from 2025-08-04 00:00:00
Test after lag features: 44,113 rows


## 3. Cross-Direction Lag

In [4]:
# 2.2 lga_arr_delay_1h — arrival congestion affecting departures
# (Inverse of lga_dep_delay_1h in arrival model)
# Uses arrival data: mean arrival delay in PREVIOUS hour

print('--- Computing lga_arr_delay_1h ---')

# Hourly aggregation of arrival delays
arr_for_lag = df_arr[['Scheduled_Arrival_Datetime', 'Arr_Calculated_Delay']].copy()
arr_for_lag = arr_for_lag.dropna()
arr_for_lag['_arr_hour'] = arr_for_lag['Scheduled_Arrival_Datetime'].dt.floor('h')

arr_hourly = arr_for_lag.groupby('_arr_hour')['Arr_Calculated_Delay'].mean().reset_index()
arr_hourly.columns = ['_match_hour', 'lga_arr_delay_1h']

# Match to departures using PREVIOUS hour (leak prevention)
for df_part, name in [(train, 'Train'), (test, 'Test')]:
    df_part['_match_hour'] = df_part['Scheduled_Departure_Datetime'].dt.floor('h') - pd.Timedelta(hours=1)
    merged = df_part.merge(arr_hourly, on='_match_hour', how='left')
    df_part['lga_arr_delay_1h'] = merged['lga_arr_delay_1h'].values
    df_part.drop(columns=['_match_hour'], inplace=True)
    print(f'  {name}: lga_arr_delay_1h mean={df_part["lga_arr_delay_1h"].mean():.2f}, '
          f'null={df_part["lga_arr_delay_1h"].isna().sum()}')

# Fill NaN with median
med = train['lga_arr_delay_1h'].median()
train['lga_arr_delay_1h'] = train['lga_arr_delay_1h'].fillna(med)
test['lga_arr_delay_1h'] = test['lga_arr_delay_1h'].fillna(med)

--- Computing lga_arr_delay_1h ---
  Train: lga_arr_delay_1h mean=8.18, null=8392
  Test: lga_arr_delay_1h mean=5.52, null=3224


## 4. Aircraft Continuity

In [5]:
# 2.3 Aircraft continuity (INVERTED: arrival delay → departure delay)
# For each departure, find the same aircraft's PREVIOUS ARRIVAL at LGA
# and use that arrival's delay as a feature

print('--- Computing aircraft continuity (inbound arrival → outbound departure) ---')

def compute_inbound_aircraft_delay(
    df_departures, df_arrivals, max_gap_hours=8.0, verbose=True
):
    """For each departure, find the same aircraft's most recent arrival."""
    deps = df_departures.copy()
    arrs = df_arrivals[['Registration', 'Scheduled_Arrival_Datetime', 'Arr_Calculated_Delay']].copy()
    arrs = arrs.dropna(subset=['Registration', 'Scheduled_Arrival_Datetime', 'Arr_Calculated_Delay'])
    arrs = arrs.sort_values('Scheduled_Arrival_Datetime')
    
    # For each departure, merge_asof to find most recent arrival of same aircraft
    deps = deps.sort_values('Scheduled_Departure_Datetime')
    
    merged = pd.merge_asof(
        deps[['Registration', 'Scheduled_Departure_Datetime']].dropna(),
        arrs.rename(columns={
            'Scheduled_Arrival_Datetime': '_prev_arr_time',
            'Arr_Calculated_Delay': '_prev_arr_delay'
        }),
        left_on='Scheduled_Departure_Datetime',
        right_on='_prev_arr_time',
        by='Registration',
        direction='backward',
        tolerance=pd.Timedelta(hours=max_gap_hours)
    )
    
    # Compute turnaround
    merged['turnaround_hours'] = (
        (merged['Scheduled_Departure_Datetime'] - merged['_prev_arr_time'])
        .dt.total_seconds() / 3600
    )
    
    # Map back to original departure index
    deps = deps.merge(
        merged[['Registration', 'Scheduled_Departure_Datetime', '_prev_arr_delay', 'turnaround_hours']],
        on=['Registration', 'Scheduled_Departure_Datetime'],
        how='left'
    )
    deps = deps.rename(columns={'_prev_arr_delay': 'prev_inbound_delay'})
    
    # Fill NaN with median
    for col in ['prev_inbound_delay', 'turnaround_hours']:
        med = deps[col].median() if deps[col].notna().any() else 0.0
        n_valid = deps[col].notna().sum()
        deps[col] = deps[col].fillna(med)
        if verbose:
            print(f'  {col}: {n_valid:,}/{len(deps):,} matched ({n_valid/len(deps)*100:.1f}%), '
                  f'mean={deps[col].mean():.2f}')
    
    return deps

# Train: use all arrivals up to cutoff
train = compute_inbound_aircraft_delay(train, arr_train, max_gap_hours=8)

# Test: use arrivals including context period
print()
test = compute_inbound_aircraft_delay(test, df_arr, max_gap_hours=8)

--- Computing aircraft continuity (inbound arrival → outbound departure) ---
  prev_inbound_delay: 87,720/104,460 matched (84.0%), mean=5.10
  turnaround_hours: 87,720/104,460 matched (84.0%), mean=1.31

  prev_inbound_delay: 37,292/44,117 matched (84.5%), mean=2.24
  turnaround_hours: 37,292/44,117 matched (84.5%), mean=1.28


## 5. Runway Config Change

In [6]:
# 2.4 Departure runway config change

print('--- Computing departure runway config change ---')

def add_dep_runway_config_change(df, datetime_col='Scheduled_Departure_Datetime',
                                  date_col='Date', rwy_col='Departure_Runway_Clean'):
    df = df.copy()
    df = df.sort_values(datetime_col).reset_index(drop=True)
    df = df.set_index(datetime_col)
    df['_rwy_shifted'] = df.groupby(date_col)[rwy_col].shift(1)
    df['_rwy_changed'] = (
        (df[rwy_col] != df['_rwy_shifted']) & df['_rwy_shifted'].notna()
    ).astype(int)
    df['dep_runway_config_change'] = df.groupby(date_col)['_rwy_changed'].transform(
        lambda x: x.rolling('1H', min_periods=1).max()
    )
    df = df.reset_index()
    df.drop(columns=['_rwy_shifted', '_rwy_changed'], inplace=True)
    df['dep_runway_config_change'] = df['dep_runway_config_change'].fillna(0)
    return df

train = add_dep_runway_config_change(train)
test = add_dep_runway_config_change(test)
print(f'  Train runway_config_change mean: {train["dep_runway_config_change"].mean():.3f}')
print(f'  Test runway_config_change mean: {test["dep_runway_config_change"].mean():.3f}')

--- Computing departure runway config change ---
  Train runway_config_change mean: 0.286
  Test runway_config_change mean: 0.251


## 6. Target Encoding

In [7]:
# 2.5 Target encoding (computed on TRAIN, mapped to TEST)

print('--- Computing target encoding on departure train set ---')
train_delay_rate = train['Is_Delayed'].mean()

# Gate delay rate
gate_target = train.groupby('Gate')['Is_Delayed'].mean()
train['dep_gate_delay_rate'] = train['Gate'].map(gate_target).fillna(train_delay_rate)
test['dep_gate_delay_rate'] = test['Gate'].map(gate_target).fillna(train_delay_rate)

# Airline delay rate
airline_target = train.groupby('Marketing Airline Desc')['Is_Delayed'].mean()
train['dep_airline_delay_rate'] = train['Marketing Airline Desc'].map(airline_target).fillna(train_delay_rate)
test['dep_airline_delay_rate'] = test['Marketing Airline Desc'].map(airline_target).fillna(train_delay_rate)

# Runway delay rate (departure runway)
runway_target = train.groupby('Departure_Runway_Clean')['Is_Delayed'].mean()
train['dep_runway_delay_rate'] = train['Departure_Runway_Clean'].map(runway_target).fillna(train_delay_rate)
test['dep_runway_delay_rate'] = test['Departure_Runway_Clean'].map(runway_target).fillna(train_delay_rate)

# === FAA features (matching arrival model pattern: NB02 Cell 10) ===
# Compute faa_delay_severity (ordinal: 0=None, 1=DD, 2=GDP, 3=GSP)
# and faa_delay_reason_raw (WX/VOL/RWY/STAFF/OTHER/None) for target encoding

for df_part in [train, test]:
    dep_dt = pd.to_datetime(df_part['Scheduled_Departure_Datetime'])
    
    df_part['faa_delay_severity'] = 0
    df_part['faa_delay_reason_raw'] = 'None'
    
    # GDP matching (window: GDP_Start → GDP_Effective_End)
    if HAS_FAA and df_gdp is not None and len(df_gdp) > 0:
        for _, row in df_gdp.iterrows():
            start, end = row.get('GDP_Start'), row.get('GDP_Effective_End')
            if pd.isna(start) or pd.isna(end):
                continue
            mask = (dep_dt >= start) & (dep_dt <= end)
            df_part.loc[mask, 'faa_delay_severity'] = np.maximum(
                df_part.loc[mask, 'faa_delay_severity'], 2)
            # Extract reason
            reason = str(row.get('GDP Reason', '')).strip().lower()
            if any(w in reason for w in ['thunder', 'storm', 'wind', 'weather', 'ceiling', 'low ceil']):
                cat = 'WX'
            elif any(w in reason for w in ['volume', 'vol']):
                cat = 'VOL'
            elif any(w in reason for w in ['runway', 'rwy']):
                cat = 'RWY'
            elif any(w in reason for w in ['staff']):
                cat = 'STAFF'
            else:
                cat = 'OTHER'
            update_mask = mask & (df_part['faa_delay_reason_raw'] == 'None')
            df_part.loc[update_mask, 'faa_delay_reason_raw'] = cat
    
    # GSP matching (window: GSP_Start → GSP_Effective_End, higher priority)
    if HAS_FAA and df_gsp is not None and len(df_gsp) > 0:
        for _, row in df_gsp.iterrows():
            start, end = row.get('GSP_Start'), row.get('GSP_Effective_End')
            if pd.isna(start) or pd.isna(end):
                continue
            mask = (dep_dt >= start) & (dep_dt <= end)
            df_part.loc[mask, 'faa_delay_severity'] = 3  # highest priority
            reason = str(row.get('GSP Reason', '')).strip().lower()
            if any(w in reason for w in ['thunder', 'storm', 'wind', 'weather', 'ceiling', 'low ceil']):
                cat = 'WX'
            elif any(w in reason for w in ['volume', 'vol']):
                cat = 'VOL'
            elif any(w in reason for w in ['runway', 'rwy']):
                cat = 'RWY'
            elif any(w in reason for w in ['staff']):
                cat = 'STAFF'
            else:
                cat = 'OTHER'
            df_part.loc[mask, 'faa_delay_reason_raw'] = cat  # GSP overwrites
    
    # DD matching (±1h window from DD_Update)
    if HAS_FAA and df_dd is not None and len(df_dd) > 0:
        window = pd.Timedelta(hours=1)
        for _, row in df_dd.iterrows():
            update_time = row.get('DD_Update')
            if pd.isna(update_time):
                continue
            mask = (dep_dt >= update_time - window) & (dep_dt <= update_time + window)
            # DD = severity 1 (lowest), only set if currently 0
            dd_mask = mask & (df_part['faa_delay_severity'] == 0)
            df_part.loc[dd_mask, 'faa_delay_severity'] = 1

print(f'  FAA severity distribution (train): {train["faa_delay_severity"].value_counts().sort_index().to_dict()}')

# FAA delay reason: target encoding
reason_target = train.groupby('faa_delay_reason_raw')['Is_Delayed'].mean()
train['dep_faa_delay_reason'] = train['faa_delay_reason_raw'].map(reason_target).fillna(train_delay_rate)
test['dep_faa_delay_reason'] = test['faa_delay_reason_raw'].map(reason_target).fillna(train_delay_rate)

print(f'  Train delay rate: {train_delay_rate:.3f}')
print(f'  Gate encoding: {len(gate_target)} gates')
print(f'  Airline encoding: {len(airline_target)} airlines')
print(f'  Runway encoding: {len(runway_target)} runways')
print(f'  FAA reason categories: {reason_target.to_dict()}')

--- Computing target encoding on departure train set ---
  FAA severity distribution (train): {0: 83951, 1: 10248, 2: 6291, 3: 3970}
  Train delay rate: 0.215
  Gate encoding: 105 gates
  Airline encoding: 12 airlines
  Runway encoding: 21 runways
  FAA reason categories: {'None': 0.1735368740644805, 'OTHER': 0.5506666666666666, 'STAFF': 0.41509433962264153, 'VOL': 0.13333333333333333, 'WX': 0.6038709677419355}


## 7. LGA Weather Features

In [8]:
# 2.6 LGA weather features (departure origin weather)

if HAS_LGA_WX and lga_wx_hourly is not None:
    print('--- Merging LGA weather ---')
    for df_part, name in [(train, 'Train'), (test, 'Test')]:
        df_part['wx_hour'] = df_part['Scheduled_Departure_Datetime'].dt.floor('h')
        n_before = len(df_part)
        merged = df_part.merge(lga_wx_hourly, on='wx_hour', how='left')
        # Copy weather columns back
        for col in lga_wx_hourly.columns:
            if col != 'wx_hour':
                df_part[col] = merged[col].values
        df_part.drop(columns=['wx_hour'], inplace=True)
        print(f'  {name}: LGA weather merged, lga_storm_flag mean={df_part.get("lga_storm_flag", pd.Series([0])).mean():.3f}')
else:
    print('Skipping LGA weather (not available)')
    for df_part in [train, test]:
        df_part['lga_storm_flag'] = 0

--- Merging LGA weather ---
  Train: LGA weather merged, lga_storm_flag mean=0.004
  Test: LGA weather merged, lga_storm_flag mean=0.002


## 8. Destination Weather

In [9]:
# 2.7 Destination weather features

if HAS_DEST_WX and dest_wx_hourly is not None:
    print('--- Merging destination weather ---')
    
    for df_part, name in [(train, 'Train'), (test, 'Test')]:
        df_part['_dest_code'] = df_part['Non-PA Airport'].str.upper().str.strip()
        df_part['wx_hour'] = df_part['Scheduled_Departure_Datetime'].dt.floor('h')
        
        # Debug: check format alignment
        if name == 'Train':
            sample_dest = df_part['_dest_code'].dropna().unique()[:5]
            sample_wx = dest_wx_hourly['dest_airport'].unique()[:5]
            print(f'  Flight dest codes:  {sample_dest}')
            print(f'  Weather airport codes: {sample_wx}')
            # Check if ICAO (KJFK) vs IATA (JFK) mismatch
            if len(sample_wx) > 0 and len(sample_dest) > 0:
                wx_len = len(str(sample_wx[0]))
                dest_len = len(str(sample_dest[0]))
                if wx_len == 4 and dest_len == 3:
                    print(f'  ⚠ ICAO/IATA mismatch detected: wx={wx_len}chars, dest={dest_len}chars')
                    print(f'  → Stripping leading K from weather airport codes')
                    dest_wx_hourly = dest_wx_hourly.copy()  # avoid mutating shared df
                    dest_wx_hourly['dest_airport'] = dest_wx_hourly['dest_airport'].str.lstrip('K')
                elif wx_len == 3 and dest_len == 4:
                    print(f'  ⚠ IATA/ICAO mismatch: adding K prefix to dest codes')
                    df_part['_dest_code'] = 'K' + df_part['_dest_code']
                else:
                    print(f'  ✓ Format aligned: wx={wx_len}chars, dest={dest_len}chars')
        
        merged = df_part.merge(
            dest_wx_hourly,
            left_on=['_dest_code', 'wx_hour'],
            right_on=['dest_airport', 'wx_hour'],
            how='left'
        )
        
        dest_wx_cols = ['dest_dewpoint', 'dest_cloud_cover', 'dest_storm_flag', 'dest_pressure_change_3h']
        for col in dest_wx_cols:
            if col in merged.columns:
                df_part[col] = merged[col].values
            else:
                df_part[col] = np.nan
        
        df_part.drop(columns=['_dest_code', 'wx_hour'], errors='ignore', inplace=True)
        if 'dest_airport' in df_part.columns:
            df_part.drop(columns=['dest_airport'], inplace=True)
        
        match_pct = merged['dest_dewpoint'].notna().mean() * 100
        print(f'  {name}: destination weather match rate={match_pct:.1f}%')
else:
    print('Skipping destination weather (not available or 0 rows)')
    dest_wx_cols = ['dest_dewpoint', 'dest_cloud_cover', 'dest_storm_flag', 'dest_pressure_change_3h']
    for df_part in [train, test]:
        for col in dest_wx_cols:
            df_part[col] = np.nan

--- Merging destination weather ---
  Flight dest codes:  ['CLE' 'DFW' 'MCO' 'CLT' 'MIA']
  Weather airport codes: ['ABE' 'ACK' 'ACY' 'ADS' 'ADW']
  ✓ Format aligned: wx=3chars, dest=3chars
  Train: destination weather match rate=33.0%
  Test: destination weather match rate=28.3%


## 9. Weather Impact Scores (V9.0)

## 9. Destination Historical Delay Rate

In [10]:
# 2.8 Destination historical delay rate (from TRAIN data only — prevents leakage)
# For each destination airport, compute average delay rate from TRAIN set, map to both train and test

dest_col = 'Non-PA Airport'
if dest_col in train.columns:
    train_delay_rate = train['Is_Delayed'].mean()
    dest_delay_rates = train.groupby(dest_col)['Is_Delayed'].mean()
    train['dest_historical_delay'] = train[dest_col].map(dest_delay_rates).fillna(train_delay_rate)
    test['dest_historical_delay'] = test[dest_col].map(dest_delay_rates).fillna(train_delay_rate)
    print(f"dest_historical_delay: {len(dest_delay_rates)} destinations, "
          f"mean={train['dest_historical_delay'].mean():.3f}, "
          f"range=[{train['dest_historical_delay'].min():.3f}, {train['dest_historical_delay'].max():.3f}]")
else:
    train['dest_historical_delay'] = 0
    test['dest_historical_delay'] = 0
    print("WARNING: dest column not found, dest_historical_delay set to 0")


dest_historical_delay: 361 destinations, mean=0.215, range=[0.000, 1.000]


In [11]:
# === V9.0: Weather Impact Scores (pre-computed, 54 airports) ===
wx_impact_path = DATA_RAW / 'WX_ImpactScores_2025_Full.csv'

if wx_impact_path.exists():
    df_wx_impact = pd.read_csv(wx_impact_path)
    df_wx_impact['Timestamp'] = (
        pd.to_datetime(df_wx_impact['date']) +
        pd.to_timedelta(df_wx_impact['hour'], unit='h')
    )
    print(f"Weather Impact Scores: {len(df_wx_impact):,} records")

    for split_name, split_df in [('train', train), ('test', test)]:
        # Build hourly Timestamp from Date + Hour
        split_df['_wx_ts'] = pd.to_datetime(split_df['Date']) + pd.to_timedelta(split_df['Hour'], unit='h')

        # dest_wx_impact
        dest_wx = df_wx_impact[['airport', 'Timestamp', 'wx_impact_score']].copy()
        dest_wx.rename(columns={'wx_impact_score': 'dest_wx_impact'}, inplace=True)
        split_df['_dest_upper'] = split_df['Non-PA Airport'].astype(str).str.upper().str.strip()
        merged = split_df[['_dest_upper', '_wx_ts']].merge(
            dest_wx, left_on=['_dest_upper', '_wx_ts'], right_on=['airport', 'Timestamp'], how='left')
        split_df['dest_wx_impact'] = merged['dest_wx_impact'].fillna(0).values

        # lga_wx_impact
        lga_wx = df_wx_impact[df_wx_impact['airport'] == 'LGA'][['Timestamp', 'wx_impact_score']].copy()
        lga_wx.rename(columns={'wx_impact_score': 'lga_wx_impact'}, inplace=True)
        lga_wx = lga_wx.drop_duplicates(subset='Timestamp', keep='first')
        merged = split_df[['_wx_ts']].merge(lga_wx, left_on='_wx_ts', right_on='Timestamp', how='left')
        split_df['lga_wx_impact'] = merged['lga_wx_impact'].fillna(0).values

        split_df.drop(columns=['_dest_upper', '_wx_ts'], inplace=True, errors='ignore')
        print(f"  {split_name}: dest_wx={split_df['dest_wx_impact'].gt(0).mean()*100:.1f}%, "
              f"lga_wx={split_df['lga_wx_impact'].gt(0).mean()*100:.1f}%")
else:
    for split_df in [train, test]:
        split_df['dest_wx_impact'] = 0
        split_df['lga_wx_impact'] = 0
    print("WARNING: WX Impact Scores not found")


Weather Impact Scores: 381,024 records
  train: dest_wx=21.3%, lga_wx=40.0%
  test: dest_wx=16.7%, lga_wx=37.1%


## 11. FAA Event Depth (V9.0)

In [12]:
# === V9.0: FAA Event Depth Features ===
faa_gdp_raw = pd.read_csv(DATA_RAW / 'LGA_FAA_GrounDelays2025.csv')
faa_gsp_raw = pd.read_csv(DATA_RAW / 'LGA_FAA_GroundStops2025.csv')

def parse_faa_times(faa_df, start_col, end_col):
    events = []
    for _, row in faa_df.iterrows():
        try:
            start = pd.to_datetime(row[start_col])
            end = pd.to_datetime(row[end_col])
            if pd.notna(start) and pd.notna(end) and end > start:
                events.append((start, end))
        except:
            continue
    return events

gdp_events = parse_faa_times(faa_gdp_raw, 'GDP Start Time', 'GDP End Time')
gsp_events = parse_faa_times(faa_gsp_raw, 'GSP Start Time', 'GSP End Time')
all_events = gdp_events + gsp_events
print(f"FAA events: GDP={len(gdp_events)}, GSP={len(gsp_events)}")

def get_faa_event_features(flight_time, events):
    active_durations = []
    for start, end in events:
        if start <= flight_time <= end:
            duration = (flight_time - start).total_seconds() / 3600
            active_durations.append(duration)
    return (max(active_durations), len(active_durations)) if active_durations else (0.0, 0)

for split_name, split_df in [('train', train), ('test', test)]:
    flight_times = pd.to_datetime(split_df['Date'].astype(str) + ' ' + split_df['Hour'].astype(str) + ':00:00', errors='coerce')
    durations, counts = [], []
    for ft in flight_times:
        if pd.isna(ft):
            durations.append(0.0); counts.append(0)
        else:
            d, c = get_faa_event_features(ft, all_events)
            durations.append(d); counts.append(c)
    split_df['faa_event_duration_hours'] = durations
    split_df['faa_active_event_count'] = counts
    print(f"  {split_name}: duration non-zero={split_df['faa_event_duration_hours'].gt(0).mean()*100:.1f}%")


FAA events: GDP=67, GSP=271
  train: duration non-zero=10.6%
  test: duration non-zero=10.8%


## 12. Cloud Ceiling & Pressure (V9.0)

In [13]:
# === V9.0: LGA Cloud Ceiling & Pressure Features ===
for split_name, split_df in [('train', train), ('test', test)]:
    if 'clds' in split_df.columns:
        split_df['lga_ceiling_low'] = split_df['clds'].isin(['OVC', 'BKN']).astype(int)
    else:
        split_df['lga_ceiling_low'] = 0
    if 'pressure_desc' in split_df.columns:
        split_df['lga_pressure_falling'] = split_df['pressure_desc'].isin(['Falling', 'Falling Rapidly']).astype(int).fillna(0)
    else:
        split_df['lga_pressure_falling'] = 0
    print(f"  {split_name}: ceiling_low={split_df['lga_ceiling_low'].mean()*100:.1f}%, pressure_falling={split_df['lga_pressure_falling'].mean()*100:.1f}%")


  train: ceiling_low=0.0%, pressure_falling=0.0%
  test: ceiling_low=0.0%, pressure_falling=0.0%


## 13. Route Risk Score (V9.0)

In [14]:
# Route risk score: destination-based delay risk
# Computed from TRAIN only (target encoding pattern — prevents leakage)
import numpy as np

dest_col = 'Non-PA Airport'
dest_stats = train.groupby(dest_col).agg(
    delay_rate=('Is_Delayed', 'mean'),
    volume=('Is_Delayed', 'count'),
    avg_delay=('Dep_Calculated_Delay', 'mean')
)
dest_stats['risk_score'] = (
    dest_stats['delay_rate'] * 
    np.log1p(dest_stats['volume']) * 
    (1 + dest_stats['avg_delay'] / dest_stats['avg_delay'].mean())
)
# Normalize to 0-1
dest_stats['risk_score'] = (
    (dest_stats['risk_score'] - dest_stats['risk_score'].min()) / 
    (dest_stats['risk_score'].max() - dest_stats['risk_score'].min())
)

train_mean_risk = dest_stats['risk_score'].mean()
train['route_risk_score'] = train[dest_col].map(dest_stats['risk_score']).fillna(train_mean_risk)
test['route_risk_score'] = test[dest_col].map(dest_stats['risk_score']).fillna(train_mean_risk)

print(f"route_risk_score: {len(dest_stats)} destinations")
print(f"  mean={train['route_risk_score'].mean():.3f}, std={train['route_risk_score'].std():.3f}")
print(f"  range=[{train['route_risk_score'].min():.3f}, {train['route_risk_score'].max():.3f}]")


route_risk_score: 361 destinations
  mean=0.306, std=0.105
  range=[0.000, 1.000]


## 14. Feature Matrix Assembly

In [15]:
# 2.9 Assemble feature matrix (V9.0: 23 features)

# V9.0 = V7.0's 21 - 3 (lga_storm_flag, dest_storm_flag, dest_cloud_cover)

#       + 2 (dest_wx_impact, lga_wx_impact) + 6 new features

feature_columns = [

    # Core Departure Lag (5)

    'delay_rate_1h',               # past 1h departure delay rate

    'delay_rolling_3h',            # past 3h departure rolling mean

    'severe_delay_count_prev',     # past 3h severe departure count

    'terminal_delay_1h',           # terminal-specific departure delay

    'dep_runway_config_change',    # departure runway changed

    

    # Cross-modal Lag (1)

    'lga_arr_delay_1h',            # arrival congestion -> departure delay

    

    # Aircraft Continuity (2)

    'prev_inbound_delay',          # previous arrival delay of same aircraft

    'turnaround_hours',            # gap between inbound arrival and outbound departure

    

    # Target Encoding (4)

    'dep_gate_delay_rate',         # gate-level delay pattern

    'dep_airline_delay_rate',      # airline delay pattern

    'dep_runway_delay_rate',       # runway delay pattern

    'dep_faa_delay_reason',        # FAA severity

    

    # Time (2)

    'Hour',

    'Month',

    

    # FAA (1)

    'faa_delay_severity',

    

    # Weather Impact Scores (2, V9.0 new)

    'dest_wx_impact',              # destination weather impact (0-10)

    'lga_wx_impact',               # LGA weather impact (37.8% non-zero)

    

    # Destination Weather (2) — retained

    'dest_dewpoint',

    'dest_pressure_change_3h',

    

    # Destination Historical (1)

    'dest_historical_delay',

    

    # === V9.0 New: Weather Impact Scores (2) ===

    # lga_wx_impact, dest_wx_impact already included above

    # === V9.0 New: FAA Event Depth (2) ===

    'faa_event_duration_hours', 'faa_active_event_count',

    # Network (1)
    'route_risk_score',            # destination route risk

    # === V9.0 New: LGA Weather (2) ===

]

# Filter to available columns

available = [c for c in feature_columns if c in train.columns]

missing = [c for c in feature_columns if c not in train.columns]

if missing:

    print(f'Missing features (will skip): {missing}')

feature_columns = available

print(f'\n=== Feature Matrix: {len(feature_columns)} features (V9.0) ===')

for i, col in enumerate(feature_columns, 1):

    print(f'  {i:2d}. {col}')

# Build X, y

X_train = train[feature_columns].copy()

y_train = train['Is_Delayed'].copy()

X_test = test[feature_columns].copy()

y_test = test['Is_Delayed'].copy()

# Fill NaN with train medians

train_medians = X_train.median()

X_train = X_train.fillna(train_medians)

X_test = X_test.fillna(train_medians)

# Regression targets

y_train_reg = train['Dep_Calculated_Delay'].copy()

y_test_reg = test['Dep_Calculated_Delay'].copy()

print(f'X_train: {X_train.shape}, y_train: {y_train.shape} (delay rate={y_train.mean():.3f})')

print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape} (delay rate={y_test.mean():.3f})')

print(f'NaN remaining: train={X_train.isna().sum().sum()}, test={X_test.isna().sum().sum()}')



=== Feature Matrix: 23 features (V9.0) ===
   1. delay_rate_1h
   2. delay_rolling_3h
   3. severe_delay_count_prev
   4. terminal_delay_1h
   5. dep_runway_config_change
   6. lga_arr_delay_1h
   7. prev_inbound_delay
   8. turnaround_hours
   9. dep_gate_delay_rate
  10. dep_airline_delay_rate
  11. dep_runway_delay_rate
  12. dep_faa_delay_reason
  13. Hour
  14. Month
  15. faa_delay_severity
  16. dest_wx_impact
  17. lga_wx_impact
  18. dest_dewpoint
  19. dest_pressure_change_3h
  20. dest_historical_delay
  21. faa_event_duration_hours
  22. faa_active_event_count
  23. route_risk_score
X_train: (104460, 23), y_train: (104460,) (delay rate=0.215)
X_test:  (44117, 23), y_test:  (44117,) (delay rate=0.178)
NaN remaining: train=0, test=0


## 15. Save

In [16]:
# === Save featured data for 03/04 ===
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'
train.to_parquet(DATA_PROCESSED / 'dep_train_featured.parquet', index=False)
test.to_parquet(DATA_PROCESSED / 'dep_test_featured.parquet', index=False)
print(f'Saved: train={len(train)} ({len(feature_columns)} features), test={len(test)}')


Saved: train=104460 (23 features), test=44117
